# Recipe Recommender Project

**Author:** *Carissa Mason*  
**Dataset:** [Kaggle – RecipeNLG Dataset](https://www.kaggle.com/datasets/paultimothymooney/recipenlg)

---

### Project Overview
A user-friendly web app that recommends recipes based on the ingredients you have! Built using Python, this app lets you:

- Find recipes from a cleaned dataset
- Filter for vegetarian-only meals
- View full step-by-step directions
- Get clean, readable outputs
---

In [1]:
import pandas as pd

# Load the dataset
df = pd.read_csv('cleaned.csv')

In [2]:
df.head(5)

# Show full columns
pd.set_option('display.max_colwidth', None)

In [3]:
df.columns

Index(['title', 'ingredients', 'directions', 'ingredients_clean',
       'directions_clean', 'vegetarian'],
      dtype='object')

In [28]:
import re

# Keep only the necessary columns
df = df[['title', 'ingredients', 'directions']].dropna()

# Clean the title casing
def fix_title_case(text):
    if isinstance(text, str):
        fixed = text.title()
        return re.sub(r"'S\b", "'s", fixed)
    return text

df['title'] = df['title'].apply(fix_title_case)

# Clean ingredients and directions (bracket/quote fix)
def clean_list_column(text):
    try:
        items = ast.literal_eval(text)
        if isinstance(items, list):
            return ", ".join(item.strip().strip('"\'') for item in items)
    except Exception as e:
        print(f"[ERROR PARSING]: {text[:60]}... → {e}")
    
    # Fallback (so it will clean it manually if it’s not a list)
    return re.sub(r'[\[\]\"]', '', str(text))

df['ingredients_clean'] = df['ingredients'].apply(clean_list_column)
df['directions_clean'] = df['directions'].apply(clean_list_column)

# Show full columns
pd.set_option('display.max_colwidth', None)

# Preview the cleaned result
df[['title', 'ingredients_clean', 'directions_clean']].head(3)


,title,ingredients_clean,directions_clean
0,No-Bake Nut Cookies,"1 c. firmly packed brown sugar, 1/2 c. evaporated milk, 1/2 tsp. vanilla, 1/2 c. broken nuts (pecans), 2 Tbsp. butter or margarine, 3 1/2 c. bite size shredded rice biscuits","In a heavy 2-quart saucepan, mix brown sugar, nuts, evaporated milk and butter or margarine., Stir over medium heat until mixture bubbles all over top., Boil and stir 5 minutes more. Take off heat., Stir in vanilla and cereal; mix well., Using 2 teaspoons, drop and shape into 30 clusters on wax paper., Let stand until firm, about 30 minutes."
1,Jewell Ball's Chicken,"1 small jar chipped beef, cut up, 4 boned chicken breasts, 1 can cream of mushroom soup, 1 carton sour cream","Place chipped beef on bottom of baking dish., Place chicken on top of beef., Mix soup and cream together; pour over chicken. Bake, uncovered, at 275° for 3 hours."
2,Creamy Corn,"2 (16 oz.) pkg. frozen corn, 1 (8 oz.) pkg. cream cheese, cubed, 1/3 c. butter, cubed, 1/2 tsp. garlic powder, 1/2 tsp. salt, 1/4 tsp. pepper","In a slow cooker, combine all ingredients. Cover and cook on low for 4 hours or until heated through and cheese is melted. Stir well before serving. Yields 6 servings."


In [30]:
# Vegetarian Filter

# Define a list of common non-vegetarian ingredients
non_veg_keywords = ['chicken', 'beef', 'pork', 'fish', 'shrimp', 'bacon', 'turkey', 'lamb', 'ham', 'sausage']

# Create vegetarian flag
def is_vegetarian(ingredients):
    return not any(meat in ingredients for meat in non_veg_keywords)

df['vegetarian'] = df['ingredients_clean'].apply(is_vegetarian)

In [31]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Vectorize
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df['ingredients_clean'])

# Build recommender function
def recommend_recipes(user_ingredients, top_n=5, vegetarian_only=False):
    # Omit recipes without proper measurements
    measurement_pattern = r"\b(?:\d+\s?/\s?\d+|\d+(?:\.\d+)?)(?:\s?)(?:cup|cups|tbsp|tablespoon|tsp|teaspoon|oz|ounce|g|gram|kg|pound|lb|ml|l|liter|pinch|dash)\b"
    
    # Filter for recipes with measurements
    filtered_df = df[df['ingredients_clean'].str.contains(measurement_pattern, flags=re.IGNORECASE, na=False)]

    if vegetarian_only:
        filtered_df = filtered_df[filtered_df['vegetarian']]

    user_vec = vectorizer.transform([user_ingredients.lower()])
    similarity_scores = cosine_similarity(user_vec, vectorizer.transform(filtered_df['ingredients_clean']))
    top_indices = similarity_scores[0].argsort()[::-1]
    
    results = filtered_df.iloc[top_indices]
    return results[['title', 'ingredients_clean', 'directions_clean']].head(top_n)

In [32]:
# Show recommendations in a clean way
import re
import ast

def show_recommendations(results):
    for i, row in results.iterrows():
        print(f"\n🍽️ {row['title']}")
        print(f"🧂 Ingredients: {row['ingredients_clean']}")
        print("📖 Directions:")

        try:
            steps = ast.literal_eval(row['directions_clean'])
            if isinstance(steps, list):
                for step in steps:
                    for sentence in step.split("."):
                        # Cleaning each sentence here
                        sentence = sentence.strip()
                        sentence = re.sub(r'^[\s\(\)\{\}\.,:;!?\"\'-]+', '', sentence)
                        if sentence:
                            print(f"- {sentence}.")
            else:
                raise ValueError("Not a list")
        except:
            for sentence in str(row['directions_clean']).split("."):
                sentence = sentence.strip()
                sentence = re.sub(r'^[\s\(\)\{\}\.,:;!?\"\'-]+', '', sentence)
                if sentence:
                    print(f"- {sentence}.")
# Example input
input_ingredients = "garlic, chicken, rice"

# Get recommendations
recommendations = recommend_recipes(input_ingredients)

In [33]:
show_recommendations(recommendations)


🍽️ Microwave Rice
🧂 Ingredients: 1 1/2 c. rice, 1/2 tsp. salt, 3 c. water
📖 Directions:
- Use 1 part rice to 2 parts water.
- Wash rice and drain.
- Add water and salt.
- Cover microwave dish or Pyrex dish with paper towel.
- Place in microwave and cook on High for 18 minutes.
- Let set for 4 to 5 minutes.

🍽️ Rice-Chicken Casserole
🧂 Ingredients: 1 whole chicken, 1 c. chopped celery, 1 Tbsp. chopped onion, 3 hard-boiled eggs, chopped, 1 1/2 c. cooked rice (1/2 c. rice cooked in 1 1/4 c. chicken broth), 1/2 c. chicken broth, 1/2 tsp. salt and pepper, 1 can cream of chicken soup, 1/2 c. mayonnaise, 1 can French fried onion rings
📖 Directions:
- Cut yellow tendons at top of legs (to keep from kicking batter off when cooking).
- Soak in cold, salty water for 1 hour.
- Coat with bread crumbs.
- Let set for 1 hour in refrigerator.
- Remove and fry in hot grease until brown.

🍽️ Chicken Casserole
🧂 Ingredients: 1 stick butter, 4 to 6 chicken breasts, 1 can cream of chicken soup, 1/2 c. mayo

In [9]:
df_trimmed = df.head(20000)
df_trimmed.to_csv("cleaned.csv", index=False)